In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys, os
sys.path.append(os.getcwd())

In [0]:
from insurer import Insurer
import scr, report

r = scr.run(Insurer.load())
print(f"SCR: {r['scr']/1e6:.1f}m   Solvency ratio: {r['solvency_ratio']:.0%}")
display(report.results_table(r))

In [0]:
display(report.sensitivities())

In [0]:
import re

def clean_cols(df):
    df = df.copy()
    df.columns = [re.sub(r"[^0-9a-zA-Z]+", "_", c).strip("_").lower() for c in df.columns]
    return df

buildup = clean_cols(report.results_table(r))
sens = clean_cols(report.sensitivities())

print(list(buildup.columns))   # check: no spaces
print(list(sens.columns))

spark.createDataFrame(buildup).write.mode("overwrite").saveAsTable("workspace.default.scr_buildup")
spark.createDataFrame(sens).write.mode("overwrite").saveAsTable("workspace.default.scr_sensitivities")
print("Saved")

In [0]:
%sql
SELECT * FROM workspace.default.scr_buildup